In [ ]:
from audiospylt.audio_utils import load_audio_sample
import numpy as np

# Parameters: wav_source (local path or URL), desired_sample_rate (None keeps native),
# convert_to_mono (bool), show_waveform (bool plot), play_audio (bool player)
# Returns: audio_data, sample_rate, audio_info (num_channels, duration, wav_filename, playback_audio, player)
# Adjustable variables
wav_source = 'source_audio/bongo.wav'
desired_sample_rate = None
convert_to_mono = True
show_waveform = True
play_audio = True

audio_data, sample_rate, audio_info = load_audio_sample(
    wav_source=wav_source,
    desired_sample_rate=desired_sample_rate,
    convert_to_mono=convert_to_mono,
    show_waveform=show_waveform,
    play_audio=play_audio,
)

playback_audio = audio_info["playback_audio"]

WAV file loaded from local path: c:\Users\egorp\Nextcloud\code\public_repos\audiospylt\notebooks\source_audio\bongo.wav
Number of audio channels: 1
Sampling rate: 44100 Hz
WAV file loaded from source_audio/bongo.wav
Total duration: 1.689 seconds


In [2]:
from audiospylt.audio_utils import trim_and_fade_audio, ensure_finite_audio
from audiospylt.generate_wave_file import render_audio

player = True
save_audio = True

# trim_and_fade_audio parameters:
# audio_data, sample_rate, num_channels, duration, wav_filename,
# start_time (s), end_time (s), add_fades (bool), fade_in_duration (s),
# fade_out_duration (s), fade_in_exponent, fade_out_exponent

start_time = 0.4
end_time = 1.2
add_fades = True
fade_in_duration = 0.04
fade_out_duration = 0.2
fade_in_exponent = 0.8
fade_out_exponent = 1.5

cut_audio_data, cut_duration = trim_and_fade_audio(
    audio_data,
    sample_rate,
    audio_info["num_channels"],
    audio_info["duration"],
    audio_info["wav_filename"],
    start_time,
    end_time,
    add_fades,
    fade_in_duration,
    fade_out_duration,
    fade_in_exponent,
    fade_out_exponent,
)

# Guard against rare NaN/Inf samples upstream so all downstream steps (analysis/export) stay stable.
cut_audio_data = ensure_finite_audio(cut_audio_data, name="cut_audio_data")

render_audio(
    cut_audio_data,
    sample_rate,
    fs_target_name="44.1kHz",
    bit_rate=24,
    save_audio=save_audio,
    player=player,
    sanitize=True,
    verbose=True,
)


[2025-12-15 16:27:34] 24-bit wave file with 44.1kHz sampling rate saved successfully to: c:\Users\egorp\Nextcloud\code\public_repos\audiospylt\notebooks\rendered_audio\generated_wave_file_44.1kHz_24bit_16_27_34.wav


'c:\\Users\\egorp\\Nextcloud\\code\\public_repos\\audiospylt\\notebooks\\rendered_audio\\generated_wave_file_44.1kHz_24bit_16_27_34.wav'

In [3]:
from audiospylt.audio_utils import plot_spectrogram

# Spectrogram parameters (allowed ranges/examples inline):
# y_axis_mode: 'linear' | 'log' | 'mel' | 'mixed'
# y_axis_mix: [0,1] only used when y_axis_mode='mixed' (0=linear spacing, 1=log-like spacing)
# mixed_log_floor_hz: >0 only used when y_axis_mode='mixed' (prevents log singularity near 0 Hz)
# n_fft: >0 (e.g., 1024/2048/4096)
# window_type: scipy window name (e.g., 'hann','hamming','blackman')
# overlap: [0, <0.95) fraction of window (0.75 = 75% overlap)
# oversample_factor: >=1.0 zero-padding multiplier (1.0 = none)
# mel_bins: >0 (typical 64/128/256) [used in 'mel' mode]
# mel_fmax: (0, Nyquist] [used in 'mel' mode]
# scaling: 'density' | 'spectrum'
# mode: 'magnitude' | 'psd'
# cmap: any Plotly colormap name
# show_plot: True/False to display

y_axis_mode = "mixed"         # choose 'linear' | 'log' | 'mel' | 'mixed'
y_axis_mix = 0.5              # 0..1 (only for 'mixed')
mixed_log_floor_hz = 1.0     # optional (only for 'mixed')

n_fft = 2048                  # FFT size; larger -> finer freq, coarser time
window_type = "hann"          # window shape
overlap = 0.5                # keep <0.95
oversample_factor = 2.0       # >=1.0; 2.0 pads FFT
mel_bins = 128                # mel bands (only for 'mel')
mel_fmax = sample_rate / 2    # cap at Nyquist (only for 'mel')
scaling = "density"           # power scaling
mode = "magnitude"            # or 'psd'
cmap = "Magma"               # Plotly colormap name; e.g. "Magma" or "Viridis"
show_plot = True              # display figure

fig, spec_info = plot_spectrogram(
    cut_audio_data,
    sample_rate,
    y_axis_mode=y_axis_mode,
    y_axis_mix=y_axis_mix,
    mixed_log_floor_hz=mixed_log_floor_hz,
    n_fft=n_fft,
    window_type=window_type,
    overlap=overlap,
    oversample_factor=oversample_factor,
    mel_bins=mel_bins,
    mel_fmax=mel_fmax,
    scaling=scaling,
    mode=mode,
    cmap=cmap,
    show=show_plot,
)

In [4]:
from audiospylt.dft_analysis import analyze_signal
from audiospylt.audio_utils import ensure_finite_audio
import numpy as np
import pandas as pd

# Analyze multiple successive time windows inside `cut_audio_data`.
# Times are in seconds, relative to the START of `cut_audio_data` (not the original file).
time_intervals = [(0, 0.10), (0.10, 0.20), (0.20, 0.40), (0.40, 0.75)]

# Peak-picking parameters (keep within sensible bounds):
# - window_type: scipy window name
# - thresh_amp_low/high: peak height bounds in *linear FFT magnitude* units (0 < low < high)
# - thresh_freq_low/high: keep peaks within [low, high] Hz (high <= Nyquist or None)
# - prominence: (advanced) raw scipy prominence in same units as the spectrum amplitude
# - prominence_rel: (recommended) [0,1] prominence as fraction of max(spec)
# - width: (advanced) raw scipy width in FFT bins
# - width_hz: (recommended) width in Hz (converted to bins)
# - distance_hz: (optional) min spacing between peaks in Hz (helps avoid picking adjacent bins)
# - freq_axis_mode: 'linear' | 'log' | 'mel' | 'mixed'  (plot only)
# - freq_axis_mix: [0,1] only for 'mixed' (0=linear spacing, 1=log-like spacing)
# - mixed_log_floor_hz: >0 only for 'mixed' (plot only)
# - amp_axis_mode: 'linear' | 'log' | 'mixed' (plot only)
# - amp_axis_mix: [0,1] only for amp_axis_mode='mixed'
# - amp_log_floor: >0 floor used for log/mixed amplitude plotting (avoids log(0))
# - auto_plot_range: auto-zoom plot ranges based on freq/amp thresholds (+ padding)
# - freq_plot_pad_hz/freq_plot_pad_frac: padding around [thresh_freq_low, thresh_freq_high] (Hz or fraction of span)
# - amp_plot_pad/amp_plot_pad_frac: additive padding around [thresh_amp_low, thresh_amp_high] (linear/mixed)
# - amp_plot_pad_ratio: multiplicative padding for log amp axis

window_type = "boxcar"
thresh_amp_low = 2e-4
thresh_amp_high = 0.08
thresh_freq_low = 30
thresh_freq_high = 5000

# Prefer the intuitive variants:
prominence_rel = 0.0025  # try 0.01..0.10
width_hz = 3             # e.g. 5.0..30.0 to prefer broader peaks
distance_hz = 10         # e.g. 5.0..30.0 to avoid adjacent-bin peaks

# Keep these as None unless you want raw scipy behavior:
prominence = None
width = None

# Plot axis scaling:
freq_axis_mode = "linear"
freq_axis_mix = 0.5
mixed_log_floor_hz = 1.0

amp_axis_mode = "mixed"
amp_axis_mix = 0.5
amp_log_floor = 1e-12

# Auto-zoom plot ranges to your thresholds (+ padding):
auto_plot_range = True
freq_plot_pad_hz = None      # set e.g. 50 for +/-50 Hz padding
freq_plot_pad_frac = 0.05    # used when freq_plot_pad_hz is None
amp_plot_pad = None          # set e.g. 0.01 for +/-0.01 amplitude padding
amp_plot_pad_frac = 0.10     # used when amp_plot_pad is None
amp_plot_pad_ratio = 0.15    # only used when amp_axis_mode='log'

show_peaks = True

# Plot control:
# - plot_each_interval=True -> plot every time window
plot_each_interval = True

# Optional: override peak-picking settings per interval.
# You can key by interval index (0..N-1) OR by the (t0, t1) tuple.
# Any keys you provide here override the "base" variables above for that interval.
# Example:
# interval_overrides = {
#     0: {"thresh_amp_low": 2e-4, "prominence_rel": 0.0025},
#     1: {"thresh_amp_low": 5e-4, "distance_hz": 20},
#     (0.20, 0.40): {"thresh_freq_high": 5000},
# }
interval_overrides = {}

# Collect per-interval peak tables for the next cell.
peaks_by_interval = []  # list of dicts: {time_start, time_stop, peaks_df, params_used}

# If cut_audio_data is multi-channel, analyze the first channel by default.
_base = cut_audio_data
if isinstance(_base, np.ndarray) and _base.ndim > 1:
    _base = _base[0] if _base.shape[0] <= _base.shape[-1] else _base[:, 0]

cut_total_duration = float(len(_base)) / float(sample_rate)

# Base params (used unless overridden per interval)
base_params = dict(
    window_type=window_type,
    thresh_amp_low=thresh_amp_low,
    thresh_amp_high=thresh_amp_high,
    thresh_freq_low=thresh_freq_low,
    thresh_freq_high=thresh_freq_high,
    prominence=prominence,
    width=width,
    prominence_rel=prominence_rel,
    width_hz=width_hz,
    distance_hz=distance_hz,
    freq_axis_mode=freq_axis_mode,
    freq_axis_mix=freq_axis_mix,
    mixed_log_floor_hz=mixed_log_floor_hz,
    amp_axis_mode=amp_axis_mode,
    amp_axis_mix=amp_axis_mix,
    amp_log_floor=amp_log_floor,
    auto_plot_range=auto_plot_range,
    freq_plot_pad_hz=freq_plot_pad_hz,
    freq_plot_pad_frac=freq_plot_pad_frac,
    amp_plot_pad=amp_plot_pad,
    amp_plot_pad_frac=amp_plot_pad_frac,
    amp_plot_pad_ratio=amp_plot_pad_ratio,
    show_peaks=show_peaks,
)

for idx, (t0, t1) in enumerate(time_intervals):
    t0 = float(t0)
    t1 = float(t1)
    if not (np.isfinite(t0) and np.isfinite(t1)):
        continue
    if t1 <= t0:
        continue

    # Clamp to available audio.
    t0c = max(0.0, min(t0, cut_total_duration))
    t1c = max(0.0, min(t1, cut_total_duration))
    if t1c <= t0c:
        continue

    i0 = int(round(t0c * sample_rate))
    i1 = int(round(t1c * sample_rate))
    segment = _base[i0:i1]
    if segment.size == 0:
        continue

    segment = ensure_finite_audio(segment, name=f"segment[{idx}]", sanitize=True, verbose=False)

    # Merge interval-specific overrides.
    overrides = dict(interval_overrides.get(idx, {}))
    overrides.update(interval_overrides.get((t0, t1), {}))
    params = dict(base_params)
    params.update(overrides)

    peaks_df = analyze_signal(
        signal=segment,
        sr=sample_rate,
        filename=f"{audio_info['wav_filename']} [win {idx}: {t0c:.3f}-{t1c:.3f}s]",
        show_plot=bool(plot_each_interval),
        **params,
    )

    peaks_by_interval.append({
        "time_start": t0c,
        "time_stop": t1c,
        "peaks_df": peaks_df,
        "params_used": params,
    })

print(f"Analyzed {len(peaks_by_interval)} interval(s) out of {len(time_intervals)} requested.")

File name: c:\Users\egorp\Nextcloud\code\public_repos\audiospylt\notebooks\source_audio\bongo.wav [win 0: 0.000-0.100s]
Duration (s): 0.1
Sampling rate (Hz): 44100

Maximum amplitude value: 0.045124
Total number of bands: 2206
Frequency resolution (Hz): 10.0

Amplitude Threshold 1: 0.0002
Amplitude Threshold 2: 0.08
Frequency Threshold 1 (Hz): 30
Frequency Threshold 2 (Hz): 5000

Peaks:


,Frequency (Hz),Amplitude
0,80.0,0.026469
1,120.0,0.024260
2,180.0,0.026948
3,240.0,0.032479
4,280.0,0.030203
...,...,...
59,4640.0,0.000359
60,4690.0,0.000402
61,4770.0,0.000440
62,4900.0,0.000426


File name: c:\Users\egorp\Nextcloud\code\public_repos\audiospylt\notebooks\source_audio\bongo.wav [win 1: 0.100-0.200s]
Duration (s): 0.1
Sampling rate (Hz): 44100

Maximum amplitude value: 0.034028
Total number of bands: 2206
Frequency resolution (Hz): 10.0

Amplitude Threshold 1: 0.0002
Amplitude Threshold 2: 0.08
Frequency Threshold 1 (Hz): 30
Frequency Threshold 2 (Hz): 5000

Peaks:


,Frequency (Hz),Amplitude
0,80.0,0.012306
1,120.0,0.005445
2,190.0,0.002638
3,240.0,0.021715
4,280.0,0.003613
5,320.0,0.034028
6,390.0,0.005868
7,450.0,0.004523
8,500.0,0.005422
9,520.0,0.010550


File name: c:\Users\egorp\Nextcloud\code\public_repos\audiospylt\notebooks\source_audio\bongo.wav [win 2: 0.200-0.400s]
Duration (s): 0.2
Sampling rate (Hz): 44100

Maximum amplitude value: 0.008233
Total number of bands: 4411
Frequency resolution (Hz): 5.0

Amplitude Threshold 1: 0.0002
Amplitude Threshold 2: 0.08
Frequency Threshold 1 (Hz): 30
Frequency Threshold 2 (Hz): 5000

Peaks:


,Frequency (Hz),Amplitude
0,30.0,0.001384
1,75.0,0.001033
2,125.0,0.000856
3,145.0,0.000248
4,180.0,0.000952
5,190.0,0.000864
6,205.0,0.000641
7,240.0,0.008233
8,280.0,0.001074
9,320.0,0.004678


File name: c:\Users\egorp\Nextcloud\code\public_repos\audiospylt\notebooks\source_audio\bongo.wav [win 3: 0.400-0.750s]
Duration (s): 0.35
Sampling rate (Hz): 44100

Maximum amplitude value: 0.001412
Total number of bands: 7718
Frequency resolution (Hz): 2.857143

Amplitude Threshold 1: 0.0002
Amplitude Threshold 2: 0.08
Frequency Threshold 1 (Hz): 30
Frequency Threshold 2 (Hz): 5000

Peaks:


,Frequency (Hz),Amplitude
0,31.428571,0.000250
1,240.000000,0.001412
2,322.857143,0.000296
3,394.285714,0.000753


Analyzed 4 interval(s) out of 4 requested.


In [5]:
from audiospylt.io_utils import events_df_from_peaks_by_interval, save_df_tsv
from IPython.display import display

events_df = events_df_from_peaks_by_interval(peaks_by_interval)

display(events_df)
save_df_tsv(events_df, "tsv/bongo_multi.tsv")

,freq_start,freq_stop,time_start,time_stop,amp_min,amp_max
0,80.000000,80.000000,0.0,0.10,0.026469,0.026469
1,120.000000,120.000000,0.0,0.10,0.024260,0.024260
2,180.000000,180.000000,0.0,0.10,0.026948,0.026948
3,240.000000,240.000000,0.0,0.10,0.032479,0.032479
4,280.000000,280.000000,0.0,0.10,0.030203,0.030203
...,...,...,...,...,...,...
149,1430.000000,1430.000000,0.2,0.40,0.000215,0.000215
150,31.428571,31.428571,0.4,0.75,0.000250,0.000250
151,240.000000,240.000000,0.4,0.75,0.001412,0.001412
152,322.857143,322.857143,0.4,0.75,0.000296,0.000296


Data saved successfully to c:\Users\egorp\Nextcloud\code\public_repos\audiospylt\notebooks\tsv/bongo_multi.tsv at 2025-12-15 16:27:34.709412.


'tsv/bongo_multi.tsv'